# * Amazon Deals Scraper

Scraper for **Amazon Today's Deals**.

### Highlights:
- **Async Playwright API**: Uses Playwright's native `async_api` to run seamlessly inside Google Colab's asyncio kernel without loop collisions.
- **Anti-Bot Defense Evasion**: Injects stealth scripts to mask `navigator.webdriver = undefined`, uses modern desktop Chrome user-agents, and mimics real browser headers.
- **Human-like Scrolling**: Smoothly scrolls in random increments (500–850px) with humanized pauses (1.2–2.2s) to dynamically trigger lazy loading of deals.
- **High-Speed BeautifulSoup Extraction**: Obtains the fully-rendered DOM source via `await page.content()` and parses it rapidly with BeautifulSoup.
- **Multi-Page Pagination**: Automatically navigates through event sale pages (`/events/sale/2/`, `/3/`, etc.) to hit your target goal (default: 300 deals).
- **Pandas & CSV Export**: Automatically organizes extracted fields into a Pandas DataFrame and downloads the CSV with one click.

## * Install Dependencies & Chromium (with System Libraries)
Python packages and Chromium along with system libraries (`--with-deps`).

In [1]:
# Install Python packages
!pip install -q playwright beautifulsoup4 pandas lxml

# Install Chromium and all required Linux OS shared libraries (libatk, etc.)
!playwright install --with-deps chromium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 9.0 MB/s eta 0:00:00
Installing dependencies...
Get:1 https://cli.github.com/packages stable InRelease [4,685 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:9 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [113 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [3,189 kB]

## * Scraper Implementation (Async Playwright + BeautifulSoup)
This cell defines the core modular functions for stealth browser automation, humanized scrolling, and BeautifulSoup parsing.

In [2]:
import asyncio
import random
import re
import time
from urllib.parse import urljoin, urlparse, parse_qs, urlencode, urlunparse

from bs4 import BeautifulSoup
import pandas as pd
from playwright.async_api import async_playwright

# Realistic Desktop Chrome User-Agent
USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/128.0.0.0 Safari/537.36"
)

# Anti-bot stealth JavaScript injected before any webpage scripts execute
STEALTH_SCRIPT = """
Object.defineProperty(navigator, 'webdriver', { get: () => undefined });
window.chrome = { runtime: {} };
Object.defineProperty(navigator, 'languages', { get: () => ['en-US', 'en'] });
Object.defineProperty(navigator, 'plugins', { get: () => [1, 2, 3, 4, 5] });
"""

async def create_stealth_browser(playwright_instance, headless=True):
    """Initializes Chromium with anti-bot overrides and realistic context headers."""
    try:
        browser = await playwright_instance.chromium.launch(
            headless=headless,
            args=[
                "--disable-blink-features=AutomationControlled",
                "--disable-infobars",
                "--no-sandbox",
                "--disable-setuid-sandbox",
                "--disable-dev-shm-usage",
                "--window-size=1920,1080"
            ]
        )
    except Exception as e:
        err_str = str(e)
        if "shared libraries" in err_str or "TargetClosedError" in type(e).__name__:
            print("\n" + "!" * 75)
            print("ERROR: Chromium could not launch because Linux shared libraries are missing.")
            print("Please run the following command in a code cell:")
            print("    !playwright install --with-deps chromium")
            print("!" * 75 + "\n")
        raise e

    context = await browser.new_context(
        user_agent=USER_AGENT,
        viewport={"width": 1920, "height": 1080},
        locale="en-US",
        timezone_id="America/New_York",
        extra_http_headers={
            "Accept-Language": "en-US,en;q=0.9",
            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
            "sec-ch-ua": '"Chromium";v="128", "Not;A=Brand";v="24", "Google Chrome";v="128"',
            "Sec-Fetch-Dest": "document",
            "Sec-Fetch-Mode": "navigate",
            "Sec-Fetch-Site": "none",
            "Sec-Fetch-User": "?1",
            "Upgrade-Insecure-Requests": "1"
        }
    )
    page = await context.new_page()
    await page.add_init_script(STEALTH_SCRIPT)
    return browser, context, page

async def check_anti_bot_detection(page) -> bool:
    """Detects if Amazon triggered a CAPTCHA or Robot Check challenge."""
    try:
        title = (await page.title()).lower()
        if "robot check" in title or "captcha" in title:
            return True
        if await page.locator("form[action*='validateCaptcha']").count() > 0:
            return True
        body_text = await page.locator("body").inner_text()
        if "Enter the characters you see below" in body_text:
            return True
    except Exception:
        pass
    return False

async def scroll_to_lazy_load(page, max_scrolls=12) -> str:
    """Emulates natural human scrolling to trigger dynamic deal loading."""
    for i in range(1, max_scrolls + 1):
        scroll_step = random.randint(500, 850)
        await page.evaluate(f"window.scrollBy({{ top: {scroll_step}, behavior: 'smooth' }});")
        await asyncio.sleep(random.uniform(1.2, 2.2))
        is_bottom = await page.evaluate("() => (window.innerHeight + window.scrollY) >= (document.body.offsetHeight - 200);")
        if is_bottom:
            break
    await asyncio.sleep(1.5)
    return await page.content()

def build_next_page_url(base_url: str, page_number: int) -> str:
    """Generates URLs for subsequent deal pages (events or search)."""
    parsed = urlparse(base_url)
    clean_path = parsed.path.rstrip("/")
    if "/events/" in clean_path:
        parts = clean_path.split("/")
        if parts[-1].isdigit():
            parts[-1] = str(page_number)
            new_path = "/".join(parts) + "/"
        else:
            new_path = f"{clean_path}/{page_number}/"
        return urlunparse((parsed.scheme, parsed.netloc, new_path, parsed.params, parsed.query, parsed.fragment))
    qs = parse_qs(parsed.query)
    qs["page"] = [str(page_number)]
    new_query = urlencode(qs, doseq=True)
    return urlunparse((parsed.scheme, parsed.netloc, parsed.path, parsed.params, new_query, parsed.fragment))

def parse_deal_card(card, base_url="https://www.amazon.com") -> dict:
    """Extracts structured fields from an individual card with BeautifulSoup."""
    asin = card.get("data-asin", "").strip()

    # Product Link & ASIN
    link_el = (
        card.find("a", class_=re.compile(r"dcl-product-link|a-link-normal", re.I))
        or card.find("a", href=re.compile(r"/dp/|/deal/"))
        or card.find("a", href=True)
    )
    product_url = ""
    if link_el and link_el.get("href"):
        product_url = urljoin(base_url, link_el["href"])
        if not asin:
            asin_match = re.search(r"/dp/([A-Z0-9]{10})", product_url)
            if asin_match:
                asin = asin_match.group(1)
        if asin and "/dp/" in product_url:
            product_url = f"https://www.amazon.com/dp/{asin}"

    # Title
    title = ""
    title_el = (
        card.find("span", class_=re.compile(r"dcl-product-label|deal-title|title|a-size-base-plus", re.I))
        or card.find("h2")
        or card.find("a", attrs={"data-testid": "deal-card-title"})
        or card.find("a", class_=re.compile(r"title", re.I))
    )
    if title_el:
        title = title_el.get_text(" ", strip=True)
    if not title:
        img = card.find("img")
        if img and img.get("alt"):
            title = img["alt"].strip()

    if not title or len(title) < 3:
        return None

    # Deal Price
    offscreen_price = card.find("span", class_="a-offscreen")
    deal_price = offscreen_price.get_text(strip=True) if offscreen_price else ""
    if not deal_price:
        p_el = card.find("span", class_=re.compile(r"price|a-color-price", re.I))
        deal_price = p_el.get_text(strip=True) if p_el else ""

    # List Price (original)
    strike_el = (
        card.find("span", class_=re.compile(r"dcl-product-price-old|a-text-price|a-text-strike"))
        or card.find("div", class_=re.compile(r"old-price"))
    )
    list_price = ""
    if strike_el:
        off = strike_el.find("span", class_="a-offscreen")
        list_price = off.get_text(strip=True) if off else strike_el.get_text(strip=True)

    # Discount Badges
    badge = (
        card.find("div", class_=re.compile(r"_badgeContainer_|dcl-badge", re.I))
        or card.find("span", class_=re.compile(r"deal-badge|badge-text|percentage", re.I))
        or card.find("span", class_="a-badge-text")
    )
    discount = badge.get_text(" ", strip=True) if badge else ""

    # Customer Rating & Reviews
    rating_el = card.find("span", class_="a-icon-alt") or card.find("i", class_=re.compile(r"a-icon-star", re.I))
    rating = rating_el.get_text(strip=True) if rating_el else ""

    rev_el = (
        card.find("span", attrs={"aria-label": re.compile(r"ratings|reviews", re.I)})
        or card.find("a", href=re.compile(r"#customerReviews", re.I))
    )
    reviews_count = rev_el.get_text(strip=True) if rev_el else ""

    # Thumbnail Image
    img_tag = card.find("img")
    image_url = img_tag.get("src", "") if img_tag else ""

    return {
        "asin": asin,
        "title": title,
        "deal_price": deal_price,
        "list_price": list_price,
        "discount": discount,
        "rating": rating,
        "reviews_count": reviews_count,
        "product_url": product_url,
        "image_url": image_url
    }

async def scrape_amazon_deals(url="https://www.amazon.com/events/labordaysale", max_deals=300, output_csv="amazon_deals.csv") -> pd.DataFrame:
    """Main async scraper pipeline to retrieve target deals count."""
    print("=" * 75)
    print("AMAZON TODAY'S DEALS SCRAPER")
    print(f"Target URL:  {url}")
    print(f"Target Goal: {max_deals} deals")
    print("=" * 75)

    all_deals = []
    seen_keys = set()
    current_page = 1
    max_pages = 10

    async with async_playwright() as p:
        browser, context, page = await create_stealth_browser(p, headless=True)
        try:
            while len(all_deals) < max_deals and current_page <= max_pages:
                page_url = url if current_page == 1 else build_next_page_url(url, current_page)
                print(f"\n[*] [Page {current_page}] Navigating to: {page_url}")

                response = await page.goto(page_url, wait_until="domcontentloaded", timeout=60000)
                if response:
                    print(f"    [*] Response status: {response.status}")

                if await check_anti_bot_detection(page):
                    print("    [!] Bot challenge detected. Pausing 5 seconds...")
                    await asyncio.sleep(5)
                else:
                    print("    [+] Passed anti-bot check.")

                await asyncio.sleep(random.uniform(2.0, 3.5))
                html_source = await scroll_to_lazy_load(page, max_scrolls=12)

                # BeautifulSoup Rapid Extraction from page.content()
                soup = BeautifulSoup(html_source, "html.parser")
                card_selectors = [
                    "div.dcl-product", "li.a-carousel-card",
                    "div[data-testid='deal-card']",
                    "div[data-component-type='s-search-result']",
                    "div[data-asin]:not([data-asin=''])"
                ]
                elements = []
                for sel in card_selectors:
                    found = soup.select(sel)
                    if len(found) > len(elements):
                        elements = found

                new_count = 0
                for card in elements:
                    deal = parse_deal_card(card)
                    if not deal:
                        continue
                    key = deal["asin"] or deal["product_url"] or deal["title"]
                    if key in seen_keys:
                        continue
                    seen_keys.add(key)
                    all_deals.append(deal)
                    new_count += 1
                    if len(all_deals) >= max_deals:
                        break

                print(f"    [+] Extracted {new_count} new deals (Total: {len(all_deals)}/{max_deals})")
                if len(all_deals) >= max_deals:
                    print(f"\n[+] Successfully reached target goal of {max_deals} deals!")
                    break
                if new_count == 0 and current_page > 1:
                    print("    [-] No further deals found on this page. Finished.")
                    break
                current_page += 1
                await asyncio.sleep(random.uniform(2.5, 4.0))
        finally:
            await context.close()
            await browser.close()

    df = pd.DataFrame(all_deals)
    if output_csv and not df.empty:
        df.to_csv(output_csv, index=False, encoding="utf-8")
        print(f"\n[✓] Saved {len(df)} deals to '{output_csv}'")
    return df

## * Scraper Run
Scrapes up to 300 deals.

Note: Google Colab natively supports top-level `await`.

In [3]:
# Set target Amazon deals URL and target deal count
TARGET_URL = "https://www.amazon.com/events/labordaysale"  # or https://www.amazon.com/deals
MAX_DEALS = 300
OUTPUT_CSV = "amazon_deals.csv"

# In Google Colab, use top-level 'await' to execute the async scraper
deals_df = await scrape_amazon_deals(url=TARGET_URL, max_deals=MAX_DEALS, output_csv=OUTPUT_CSV)

AMAZON TODAY'S DEALS SCRAPER
Target URL:  https://www.amazon.com/events/labordaysale
Target Goal: 300 deals

[*] [Page 1] Navigating to: https://www.amazon.com/events/labordaysale
    [*] Response status: 200
    [+] Passed anti-bot check.
    [+] Extracted 113 new deals (Total: 113/300)

[*] [Page 2] Navigating to: https://www.amazon.com/events/labordaysale/2/
    [*] Response status: 200
    [+] Passed anti-bot check.
    [+] Extracted 76 new deals (Total: 189/300)

[*] [Page 3] Navigating to: https://www.amazon.com/events/labordaysale/3/
    [*] Response status: 200
    [+] Passed anti-bot check.
    [+] Extracted 15 new deals (Total: 204/300)

[*] [Page 4] Navigating to: https://www.amazon.com/events/labordaysale/4/
    [*] Response status: 200
    [+] Passed anti-bot check.
    [+] Extracted 15 new deals (Total: 219/300)

[*] [Page 5] Navigating to: https://www.amazon.com/events/labordaysale/5/
    [*] Response status: 200
    [+] Passed anti-bot check.
    [+] Extracted 15 new de

## * Preview & Inspect the Results
Inspect the extracted deals table and summary metrics.

In [4]:
print(f"Total Deals Retrieved: {len(deals_df)}\n")
cols = [c for c in ["title", "deal_price", "discount", "rating", "product_url"] if c in deals_df.columns]
deals_df[cols].head(10)

Total Deals Retrieved: 300



,title,deal_price,discount,rating,product_url
0,Sony WH-1000XM5 Premium Noise Cancelling Wirel...,$198.00,50% off Limited time deal,,https://www.amazon.com/dp/B09XS7JWHH
1,"Garmin vívoactive® 5, Health & Fitness GPS Sma...",$181.97,39% off Limited time deal,,https://www.amazon.com/dp/B0CG6NBJ61
2,Soundcore by Anker Q20i Hybrid Active Noise Ca...,$44.98,36% off Limited time deal,,https://www.amazon.com/dp/B0F4884LN3
3,"Lenovo Essential 15.6"" FHD Everyday Laptop, In...",$399.99,50% off Ends in,,https://www.amazon.com/dp/B0H9B5263J
4,Sony WH-CH520 Wireless On-Ear Bluetooth Headph...,$38.00,46% off Limited time deal,,https://www.amazon.com/dp/B0BS1RT9S2
5,TI-84 Evo Graphing Calculator Texas Instrument...,$104.99,34% off Limited time deal,,https://www.amazon.com/dp/B0G8QXXF8B
6,"Amazon Echo Dot Max (newest model), Alexa spea...",$94.99,21% off Limited time deal,,https://www.amazon.com/dp/B0D6V1H9PM
7,"Anker soundcore 2 Portable Bluetooth Speaker, ...",$29.98,33% off Limited time deal,,https://www.amazon.com/dp/B01MTB55WH
8,"Soundcore Boom 2 by Anker, 80W Outdoor Bluetoo...",$89.99,31% off Ends in,,https://www.amazon.com/dp/B0CQ53RVTW
9,Philips 24 Inch Computer Monitor FHD 100Hz VA ...,$74.99,17% off Limited time deal,,https://www.amazon.com/dp/B0C8ZKV5R9


## * Creation of CSV File
Run this cell in Google Colab to download `amazon_deals.csv` directly to your local machine.

In [5]:
try:
    from google.colab import files
    files.download(OUTPUT_CSV)
    print(f"[✓] Download triggered for {OUTPUT_CSV}")
except ImportError:
    print(f"Not running inside Google Colab. The CSV is saved locally at: {OUTPUT_CSV}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

[✓] Download triggered for amazon_deals.csv
